# LeetCode 3982 — Sum of Integers with Maximum Digit Range bằng Python

Link đề: https://leetcode.com/problems/sum-of-integers-with-maximum-digit-range/ · Độ khó: **Easy** · Chủ đề: Array, Math, Simulation

Skeleton LeetCode:

```python
class Solution:
    def maxDigitRange(self, nums: List[int]) -> int:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình. Slug URL được suy ra từ tên bài.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: **digit range** của một số là (chữ số lớn nhất) trừ (chữ số nhỏ nhất); cần **tổng các số trong `nums`** có digit range bằng **giá trị lớn nhất** của digit range.
2. Biết tính chữ số lớn nhất / nhỏ nhất của một số bằng `str(x)` + `max/min` hoặc bằng `divmod`.
3. Nắm mẫu **"một lượt, theo dõi giá trị tốt nhất và tổng của những phần tử đạt giá trị đó"**, gồm bước **đặt lại tổng** khi tìm thấy giá trị tốt hơn.
4. Viết 4 cách (hai lượt, một lượt, `groupby`/dict theo range, tính digit range bằng số học) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh lỗi: cộng digit range thay vì cộng số, quên đặt lại tổng khi gặp range lớn hơn, lấy số lớn nhất thay vì range lớn nhất, so sánh chuỗi thay vì chữ số.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. **Digit range** của một số nguyên là hiệu giữa **chữ số lớn nhất** và **chữ số nhỏ nhất** của nó. Ví dụ `5724` có chữ số lớn nhất `7`, nhỏ nhất `2` nên digit range là `7 - 2 = 5`.

Trả về **tổng** của mọi số trong `nums` có digit range **bằng digit range lớn nhất** trong mảng.

| Ví dụ | `nums` | Output | Giải thích |
|---|---|---|---|
| 1 | `[5724, 111, 350]` | `6074` | Range: `5724 -> 5`, `111 -> 0`, `350 -> 5` (`5 - 0`). Lớn nhất là `5`, các số đạt là `5724` và `350`; tổng `6074` |
| 2 | `[90, 900]` | `990` | Cả hai có range `9 - 0 = 9`, đều lớn nhất; tổng `990` |

Constraints: `1 <= len(nums) <= 100`; `10 <= nums[i] <= 10^5`.

Chú ý: đáp án là **tổng các số** (giá trị `nums[i]`), không phải tổng các digit range.

In [1]:
nums = [5724, 111, 350]
for x in nums:
    ds = [int(c) for c in str(x)]
    print(f"{x}: chữ số={ds}, max={max(ds)}, min={min(ds)}, range={max(ds) - min(ds)}")

5724: chữ số=[5, 7, 2, 4], max=7, min=2, range=5
111: chữ số=[1, 1, 1], max=1, min=1, range=0
350: chữ số=[3, 5, 0], max=5, min=0, range=5


## 3. Kiến thức nền

### 3.1. Chữ số của một số: `str(x)` và `int(c)`

`str(5724)` là `"5724"`; duyệt từng ký tự rồi đổi sang số bằng `int`. Có thể bỏ bước đổi: ký tự chữ số so sánh theo thứ tự ASCII đúng như số (`'0' < '1' < ... < '9'`), nên `max("5724")` là `'7'`.

In [2]:
x = 5724
s = str(x)
print(s, max(s), min(s), int(max(s)) - int(min(s)))

5724 7 2 5


### 3.2. `max` và `min` trên iterable

`max(iterable)`, `min(iterable)` cho phần tử lớn/nhỏ nhất, có thể truyền chuỗi, list, hoặc generator. `max` với `key=` và `default=` cũng hữu ích (không dùng ở đây).

In [3]:
print(max([3, 9, 2]), min("350"), max(len(str(v)) for v in [5, 55, 555]))

9 0 3


### 3.3. Tách chữ số bằng `divmod` (không chuỗi)

`while x: x, d = divmod(x, 10)` lấy lần lượt chữ số cuối; ta theo dõi `hi = max(hi, d)` và `lo = min(lo, d)` ngay trong vòng lặp.

In [4]:
x, hi, lo = 5724, 0, 9
while x:
    x, d = divmod(x, 10)
    hi, lo = max(hi, d), min(lo, d)
print(hi, lo, hi - lo)

7 2 5


### 3.4. Mẫu "một lượt theo dõi giá trị tốt nhất"

Duyệt từng phần tử, giữ `best` (giá trị tốt nhất đang có) và `total` (tổng các phần tử đạt `best`). Gặp giá trị **tốt hơn** -> `best = giá_trị`, `total = phần_tử` (đặt lại!). Gặp giá trị **bằng** -> `total += phần_tử`. Gặp giá trị **kém hơn** -> bỏ qua.

In [5]:
best, total = -1, 0
for x, r in [(5724, 5), (111, 0), (350, 5)]:
    if r > best:
        best, total = r, x
    elif r == best:
        total += x
    print(f"x={x}, r={r} -> best={best}, total={total}")

x=5724, r=5 -> best=5, total=5724
x=111, r=0 -> best=5, total=5724
x=350, r=5 -> best=5, total=6074


### 3.5. `defaultdict` / nhóm theo khóa

Có thể gom các số theo digit range: `groups[r].append(x)`, rồi lấy nhóm có khóa lớn nhất `groups[max(groups)]`. Nhiều bộ nhớ hơn nhưng rất rõ ràng.

In [6]:
from collections import defaultdict
groups = defaultdict(list)
for x, r in [(5724, 5), (111, 0), (350, 5)]:
    groups[r].append(x)
print(dict(groups), groups[max(groups)], sum(groups[max(groups)]))

{5: [5724, 350], 0: [111]} [5724, 350] 6074


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hai bước tách bạch:

1. Với mỗi số, tính **digit range** của nó.
2. Tìm range **lớn nhất**, rồi cộng các số có range đúng bằng đó.

Tính tay `[5724, 111, 350]`: range `5, 0, 5` -> lớn nhất `5` -> cộng `5724 + 350 = 6074`.
Tính tay `[90, 900]`: range `9, 9` -> lớn nhất `9` -> `990`.
Tính tay `[12, 34, 59]`: range `1, 1, 4` -> lớn nhất `4` -> chỉ `59` -> `59`.

| `nums` | range từng số | range lớn nhất | tổng |
|---|---|---|---|
| `[5724, 111, 350]` | 5, 0, 5 | 5 | 6074 |
| `[90, 900]` | 9, 9 | 9 | 990 |
| `[12, 34, 59]` | 1, 1, 4 | 4 | 59 |
| `[11, 22, 33]` | 0, 0, 0 | 0 | 66 |

In [7]:
def digit_range(x):
    s = str(x)
    return int(max(s)) - int(min(s))

for nums in ([5724, 111, 350], [90, 900], [12, 34, 59], [11, 22, 33]):
    rs = [digit_range(x) for x in nums]
    mx = max(rs)
    print(nums, rs, mx, sum(x for x, r in zip(nums, rs) if r == mx))

[5724, 111, 350] [5, 0, 5] 5 6074
[90, 900] [9, 9] 9 990
[12, 34, 59] [1, 1, 4] 4 59
[11, 22, 33] [0, 0, 0] 0 66


## 5. Cách 1 — Brute Force / baseline: hai lượt

1. Lượt 1: tính list `ranges` gồm digit range của mọi số.
2. `mx = max(ranges)`.
3. Lượt 2: cộng `nums[i]` nếu `ranges[i] == mx`.

Dễ hiểu, dễ gỡ lỗi (in được list `ranges`). "Nút thắt" nhỏ: duyệt hai lần và dùng một list phụ, nhưng với `n <= 100` không đáng kể.

In [8]:
class SolutionTwoPass:
    def maxDigitRange(self, nums):
        ranges = []
        for x in nums:
            s = str(x)
            ranges.append(int(max(s)) - int(min(s)))
        mx = max(ranges)
        total = 0
        for x, r in zip(nums, ranges):
            if r == mx:
                total += x
        return total

print(SolutionTwoPass().maxDigitRange([5724, 111, 350]), SolutionTwoPass().maxDigitRange([90, 900]))

6074 990


**Độ phức tạp:** Time `O(n L)` với `L` = số chữ số tối đa (`<= 6`), Space `O(n)` cho `ranges`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** gộp hai lượt thành **một lượt** bằng mẫu "giá trị tốt nhất + tổng của nhóm tốt nhất":

- `best` = digit range lớn nhất đã thấy; `total` = tổng các số có range `== best` **trong phần đã duyệt**;
- phần tử mới có range `r`:
  - nếu `r > best`: nhóm tốt nhất đổi -> `best = r`, `total = x` (**bỏ** các số cũ);
  - nếu `r == best`: `total += x`;
  - nếu `r < best`: bỏ qua.

**Invariant:** sau mỗi bước, `total` bằng tổng các số đã duyệt có digit range bằng `best`, và `best` bằng range lớn nhất của phần đã duyệt. Khởi tạo `best = -1` (nhỏ hơn mọi range hợp lệ, vì range `>= 0`) và `total = 0`.

**Chú ý:** quên đặt lại `total` khi gặp `r > best` là lỗi phổ biến nhất của mẫu này.

## 7. Các cách tiếp theo

### Cách 2 — Một lượt theo dõi `best` và `total`

Đúng theo mục 6.

In [9]:
class SolutionOnePass:
    def maxDigitRange(self, nums):
        best, total = -1, 0
        for x in nums:
            s = str(x)
            r = int(max(s)) - int(min(s))
            if r > best:
                best, total = r, x
            elif r == best:
                total += x
        return total

print(SolutionOnePass().maxDigitRange([5724, 111, 350]), SolutionOnePass().maxDigitRange([90, 900]))

6074 990


**Độ phức tạp:** Time `O(n L)`, Space `O(1)`.

### Cách 3 — Nhóm theo digit range (`defaultdict`)

Gom số theo khóa `range`, rồi lấy tổng nhóm của khóa lớn nhất. Rõ ràng, thích hợp khi sau này cần thêm thông tin từng nhóm.

In [10]:
from collections import defaultdict

class SolutionGroup:
    def maxDigitRange(self, nums):
        groups = defaultdict(int)
        for x in nums:
            s = str(x)
            groups[int(max(s)) - int(min(s))] += x
        return groups[max(groups)]

print(SolutionGroup().maxDigitRange([5724, 111, 350]), SolutionGroup().maxDigitRange([90, 900]))

6074 990


**Độ phức tạp:** Time `O(n L)`, Space `O(k)` với `k <= 10` (digit range chỉ nhận giá trị `0..9`).

### Cách 4 — Digit range bằng số học (`divmod`), không chuỗi

Tách chữ số bằng `divmod`, theo dõi `hi`, `lo`. Tránh tạo chuỗi và gọi `int`/`max` trên chuỗi.

In [11]:
def digit_range_math(x):
    hi, lo = 0, 9
    while x:
        x, d = divmod(x, 10)
        if d > hi:
            hi = d
        if d < lo:
            lo = d
    return hi - lo

class SolutionMath:
    def maxDigitRange(self, nums):
        best, total = -1, 0
        for x in nums:
            r = digit_range_math(x)
            if r > best:
                best, total = r, x
            elif r == best:
                total += x
        return total

print(SolutionMath().maxDigitRange([5724, 111, 350]), SolutionMath().maxDigitRange([90, 900]))

6074 990


**Độ phức tạp:** Time `O(n L)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 2 trên `nums = [5724, 111, 350]`:

| `x` | chữ số | range `r` | so với `best` | `best` | `total` |
|---|---|---|---|---|---|
| (đầu) | | | | -1 | 0 |
| 5724 | 5,7,2,4 | 5 | `5 > -1` -> đặt lại | 5 | 5724 |
| 111 | 1,1,1 | 0 | `0 < 5` -> bỏ qua | 5 | 5724 |
| 350 | 3,5,0 | 5 | `5 == 5` -> cộng | 5 | 6074 |

In [12]:
def trace_one_pass(nums):
    best, total = -1, 0
    print("bắt đầu: best=-1, total=0")
    for x in nums:
        s = str(x)
        r = int(max(s)) - int(min(s))
        if r > best:
            action, best, total = "r > best -> đặt lại", r, x
        elif r == best:
            action, total = "r == best -> cộng", total + x
        else:
            action = "r < best -> bỏ qua"
        print(f"  x={x:>6}: max={max(s)}, min={min(s)}, r={r} | {action:<22} -> best={best}, total={total}")
    return total

assert trace_one_pass([5724, 111, 350]) == 6074
assert trace_one_pass([90, 900]) == 990
assert trace_one_pass([12, 34, 59, 10]) == 59

bắt đầu: best=-1, total=0
  x=  5724: max=7, min=2, r=5 | r > best -> đặt lại    -> best=5, total=5724
  x=   111: max=1, min=1, r=0 | r < best -> bỏ qua     -> best=5, total=5724
  x=   350: max=5, min=0, r=5 | r == best -> cộng      -> best=5, total=6074
bắt đầu: best=-1, total=0
  x=    90: max=9, min=0, r=9 | r > best -> đặt lại    -> best=9, total=90
  x=   900: max=9, min=0, r=9 | r == best -> cộng      -> best=9, total=990
bắt đầu: best=-1, total=0
  x=    12: max=2, min=1, r=1 | r > best -> đặt lại    -> best=1, total=12
  x=    34: max=4, min=3, r=1 | r == best -> cộng      -> best=1, total=46
  x=    59: max=9, min=5, r=4 | r > best -> đặt lại    -> best=4, total=59
  x=    10: max=1, min=0, r=1 | r < best -> bỏ qua     -> best=4, total=59


In [13]:
def trace_reset_needed(nums):
    # cho thấy vì sao phải đặt lại total khi tìm thấy range lớn hơn
    best, total = -1, 0
    for x in nums:
        r = digit_range(x)
        if r > best:
            print(f"  x={x}: range {r} > best {best}: bỏ total cũ {total}, bắt đầu lại từ {x}")
            best, total = r, x
        elif r == best:
            total += x
    return total

assert trace_reset_needed([11, 22, 19]) == 19

  x=11: range 0 > best -1: bỏ total cũ 0, bắt đầu lại từ 11
  x=19: range 8 > best 0: bỏ total cũ 33, bắt đầu lại từ 19


## 9. Code nộp LeetCode

Dùng Cách 2 (một lượt, `O(1)` bộ nhớ phụ).

In [14]:
from typing import List, Optional


class Solution:
    def maxDigitRange(self, nums: List[int]) -> int:
        best, total = -1, 0
        for x in nums:
            s = str(x)
            r = int(max(s)) - int(min(s))
            if r > best:
                best, total = r, x
            elif r == best:
                total += x
        return total


print(Solution().maxDigitRange([5724, 111, 350]), Solution().maxDigitRange([90, 900]))

6074 990


## 10. Vì sao thuật toán đúng

Ký hiệu `R(x)` là digit range của `x` và `M = max_i R(nums[i])`. Đáp án mong muốn là `sum{ nums[i] : R(nums[i]) = M }`.

1. **Digit range đúng.** `max(str(x))` là ký tự chữ số lớn nhất, `min(str(x))` là nhỏ nhất; vì `'0' < ... < '9'` theo ASCII giống thứ tự số, nên `int(max) - int(min) = R(x)`.
2. **Invariant** (Cách 2): sau khi xử lý `k` phần tử đầu, `best = max_{i<k} R(nums[i])` và `total = sum{ nums[i] : i < k, R(nums[i]) = best }`.
   - Khởi tạo (`k = 0`): `best = -1`, `total = 0` (tổng rỗng).
   - Bước: phần tử mới có `r = R(x)`. Nếu `r > best`, range lớn nhất mới là `r` và chỉ phần tử mới đạt nó (các phần tử cũ có range `<= best < r`), nên `total = x`. Nếu `r == best` thì `total += x`. Nếu `r < best`, phần tử mới không thuộc nhóm lớn nhất, giữ nguyên.
3. **Kết luận.** Khi `k = n`: `best = M` và `total` là đáp án.
4. Cách 1 làm đúng định nghĩa (tính `M` rồi cộng), Cách 3 gom theo từng range rồi lấy khóa lớn nhất, Cách 4 chỉ đổi cách tính `R(x)`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums) <= 100`, `L` = số chữ số tối đa của một phần tử (`<= 6` vì `nums[i] <= 10^5`).

- Cách 1: Time `O(n L)`, Space `O(n)`.
- Cách 2 (nộp): Time `O(n L)`, Space `O(1)`.
- Cách 3: Time `O(n L)`, Space `O(1)` (tối đa 10 khóa).
- Cách 4: Time `O(n L)`, Space `O(1)`.

Vì `L` là hằng số nhỏ nên coi như `O(n)`. Phải đọc mọi chữ số của mọi số nên không thể tốt hơn `O(n L)`.

## 12. Bộ test

### Cách kiểm thử trong notebook này

Chúng ta dùng ba lớp kiểm thử, từ dễ đến khó:

1. **Test cố định**: ví dụ của đề + các trường hợp biên tự nghĩ ra. Mỗi test là một `assert hàm(input) == kết_quả_mong_đợi`. Nếu sai, Python ném `AssertionError` và dừng ngay, kèm thông tin ta in ra để dễ lần lại.
2. **Test ngẫu nhiên có `random.seed`**: gọi `random.seed(số)` trước vòng lặp để **mỗi lần chạy lại sinh đúng cùng dữ liệu** (tái lập được). Input luôn được sinh **trong đúng constraints** của đề.
3. **Đối chiếu nhiều cách (oracle)**: một cách rất đơn giản, hiển nhiên đúng (brute force) đóng vai "trọng tài"; mọi cách nhanh hơn phải cho cùng kết quả với trọng tài trên mọi input ngẫu nhiên.

In [15]:
import random

solutions = [SolutionTwoPass(), SolutionOnePass(), SolutionGroup(), SolutionMath(), Solution()]

fixed = [
    ([5724, 111, 350], 6074), ([90, 900], 990), ([12, 34, 59], 59), ([11, 22, 33], 66),
    ([10], 10), ([99999], 99999), ([100000], 100000), ([10, 11], 10), ([11, 10], 10),
    ([19, 91, 55, 100], 19 + 91), ([123, 321, 213, 555], 123 + 321 + 213),
    ([20, 200, 2000, 20000, 100000], None),
]
for nums, expected in fixed:
    if expected is None:
        rs = [digit_range(x) for x in nums]
        expected = sum(x for x, r in zip(nums, rs) if r == max(rs))
    for sol in solutions:
        assert sol.maxDigitRange(list(nums)) == expected, (type(sol).__name__, nums)
print("test cố định OK:", len(fixed), "ca")

def ref(nums):
    rs = [digit_range(x) for x in nums]
    return sum(x for x, r in zip(nums, rs) if r == max(rs))

random.seed(3982)
for _ in range(600):
    n = random.randint(1, 100)
    hi = random.choice([99, 999, 10 ** 5])
    nums = [random.randint(10, hi) for _ in range(n)]
    expected = ref(nums)
    for sol in solutions:
        assert sol.maxDigitRange(list(nums)) == expected, (type(sol).__name__, nums)
print("test ngẫu nhiên OK")

test cố định OK: 12 ca
test ngẫu nhiên OK


### Test thuộc tính (property test)

(1) Hoán vị `nums` không đổi kết quả. (2) Đáp án nằm giữa `min(nums)` và `sum(nums)` và là tổng của một **tập con** của `nums`. (3) Nếu mọi số đều có range `0` (ví dụ `11, 22, 333`) thì đáp án bằng `sum(nums)`. (4) Lặp mỗi phần tử hai lần nhân đôi đáp án.

In [16]:
random.seed(39820)
for _ in range(400):
    n = random.randint(1, 30)
    nums = [random.randint(10, 10 ** 5) for _ in range(n)]
    r = Solution().maxDigitRange(nums)
    sh = nums[:]; random.shuffle(sh)
    assert Solution().maxDigitRange(sh) == r
    assert min(nums) <= r <= sum(nums)
    assert Solution().maxDigitRange(nums + nums) == 2 * r
for nums in ([11, 22, 333], [55, 7777, 99]):
    assert Solution().maxDigitRange(nums) == sum(nums)
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: cộng **digit range** thay vì cộng số

Đề hỏi tổng **các số** `nums[i]`. Cộng range cho `5 + 5 = 10`.

In [17]:
def wrong_sum_ranges(nums):
    rs = [digit_range(x) for x in nums]
    return sum(r for r in rs if r == max(rs))                 # SAI: cộng range

print(wrong_sum_ranges([5724, 111, 350]), "!=", Solution().maxDigitRange([5724, 111, 350]))
assert wrong_sum_ranges([5724, 111, 350]) == 10 and Solution().maxDigitRange([5724, 111, 350]) == 6074

10 != 6074


### Lỗi 2: quên đặt lại `total` khi gặp range lớn hơn

Nếu `r > best` mà chỉ cập nhật `best` (không đặt `total = x`) thì tổng cũ (của range nhỏ hơn) bị lẫn vào.

In [18]:
def wrong_no_reset(nums):
    best, total = -1, 0
    for x in nums:
        r = digit_range(x)
        if r >= best:                                          # SAI: gộp cả > và ==, không đặt lại
            best = r
            total += x
    return total

print(wrong_no_reset([111, 350]), "!=", Solution().maxDigitRange([111, 350]))
assert wrong_no_reset([111, 350]) == 461 and Solution().maxDigitRange([111, 350]) == 350

461 != 350


### Lỗi 3: lấy **số lớn nhất** thay vì **digit range lớn nhất**

`max(nums)` không liên quan tới range. `[90, 12345]`: số lớn nhất `12345` (range 4) nhưng `90` có range `9`.

In [19]:
def wrong_max_value(nums):
    m = max(nums)
    return m                                                   # SAI: nhầm đại lượng

print(wrong_max_value([90, 12345]), "!=", Solution().maxDigitRange([90, 12345]))
assert wrong_max_value([90, 12345]) == 12345 and Solution().maxDigitRange([90, 12345]) == 90

12345 != 90


### Lỗi 4: so sánh **chuỗi** thay vì số khi tính range

`max(str(x)) - min(str(x))` trừ hai **chuỗi** gây `TypeError`. Phải đổi qua `int`.

In [20]:
def wrong_str_minus(x):
    s = str(x)
    return max(s) - min(s)                                     # SAI: trừ hai chuỗi

try:
    wrong_str_minus(5724)
except TypeError as e:
    print("TypeError:", e)
assert digit_range(5724) == 5

TypeError: unsupported operand type(s) for -: 'str' and 'str'


### Lỗi 5: nhầm "chữ số lớn nhất trừ nhỏ nhất" với "số lớn nhất trừ nhỏ nhất của mảng"

Digit range tính **trong từng số**, không phải giữa các số của mảng. `[10, 100]`: hiệu giữa hai số là `90`, nhưng đề hỏi range của từng số: `1` và `1`.

In [21]:
def wrong_array_range(nums):
    return max(nums) - min(nums)                               # SAI: đây là phạm vi của mảng

print(wrong_array_range([10, 100]), "!=", digit_range(10), digit_range(100))
assert wrong_array_range([10, 100]) == 90 and digit_range(10) == digit_range(100) == 1
assert Solution().maxDigitRange([10, 100]) == 110

90 != 1 1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng các phần tử đạt giá trị **lớn nhất** của hàm `f`" -> mẫu một lượt `best`/`total` với **đặt lại khi gặp giá trị tốt hơn**. Mẫu này cũng xuất hiện ở "đếm số phần tử bằng giá trị lớn nhất" (đếm thay tổng).
- Digit range = `max(str(x)) - min(str(x))` bằng ký tự rồi `int` (so sánh ký tự chữ số đúng thứ tự số).
- Khởi tạo `best = -1` khi hàm `f >= 0`, để phần tử đầu tiên luôn kích hoạt nhánh "tốt hơn".
- Hai cách chắc ăn khi nghi ngờ: viết **bản hai lượt** (trọng tài) và so với bản một lượt.
- Chữ số chỉ `0..9` nên digit range nằm trong `0..9`: có thể dùng mảng 10 ô thay cho từ điển.
- Đọc kỹ đề: tổng các **số**, không phải tổng range.

## 15. Biến thể của bài

1. **Count Elements With Maximum Frequency** (3005), **Sum of Digits of String After Convert** (1945): đếm/tổng theo giá trị tốt nhất.
2. **Maximum Difference Between Increasing Elements** (2016): hiệu lớn nhất.
3. **Largest Number After Digit Swaps by Parity** (2231): thao tác chữ số.
4. Biến thể: tổng các số có digit range **nhỏ nhất**; hoặc trả về **số** có digit range lớn nhất (nhỏ nhất nếu có nhiều).
5. Biến thể: dùng "digit range" khác (tổng chữ số, tích chữ số, số chữ số khác nhau).
6. Biến thể: `nums` rất lớn (`10^5` phần tử): vẫn một lượt, nhưng nên dùng cách `divmod` hoặc bảng tra cho nhanh.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai lượt | tính mọi range, lấy max, cộng | `O(nL)` | `O(n)` | Dễ gỡ lỗi, trọng tài |
| 2. Một lượt | `best`/`total` đặt lại khi tốt hơn | `O(nL)` | `O(1)` | **Nộp bài** |
| 3. Nhóm theo range | `defaultdict`, lấy khóa lớn nhất | `O(nL)` | `O(1)` (<=10 khóa) | Cần thêm thông tin từng nhóm |
| 4. Số học | `divmod` thay chuỗi | `O(nL)` | `O(1)` | Tránh tạo chuỗi, nhanh |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().maxDigitRange([29, 92, 55])` và `Solution().maxDigitRange([100, 99])` ra gì?

**Bài 2 (tính tay):** với `[4071, 8888, 1090, 369]`, tính range từng số rồi đáp án.

**Bài 3 (điền TODO):** hoàn thành `min_digit_range_sum(nums)`: tổng các số có digit range **nhỏ nhất**.

**Bài 4 (tự viết lại):** viết `number_with_max_range(nums)`: trả về **số đầu tiên** trong `nums` có range lớn nhất.

**Bài 5 (mini challenge):** viết `count_max_range(nums)`: đếm bao nhiêu số đạt range lớn nhất (mẫu một lượt với `count`).

In [22]:
# Bài 1: đoán rồi chạy
print(Solution().maxDigitRange([29, 92, 55]), Solution().maxDigitRange([100, 99]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def min_digit_range_sum_todo(nums):
    # TODO: giống một lượt best/total nhưng đổi dấu so sánh (nhỏ hơn thì đặt lại)
    return 0

121 100


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: range của `29` là `9 - 2 = 7`, của `92` cũng `7`, của `55` là `0`; `100` có range `1 - 0 = 1`, `99` có range `0`. Bài 2: tính range của từng số.

**Mức 2 (vừa):** Bài 3: khởi tạo `best = 10` (lớn hơn mọi range) và đặt lại khi `r < best`. Bài 4: lưu `(range, số)` khi `r > best` (chỉ `>`, không `>=`, để giữ số đầu tiên). Bài 5: `count` đặt lại bằng `1` khi `r > best`, `+= 1` khi bằng.

**Mức 3 (gần đáp án):** Bài 1: `29 + 92 = 121` và `100`. Bài 2: range `7, 0, 9, 6` -> đáp án `1090`. Bài 4: `[15, 29, 92]` có range `4, 7, 7` nên số đầu tiên đạt range lớn nhất là `29`. Bài 5: `[29, 92, 55] -> 2`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert Solution().maxDigitRange([29, 92, 55]) == 121 and Solution().maxDigitRange([100, 99]) == 100
assert [digit_range(x) for x in [4071, 8888, 1090, 369]] == [7, 0, 9, 6]
assert Solution().maxDigitRange([4071, 8888, 1090, 369]) == 1090

# Bài 3
def min_digit_range_sum(nums):
    best, total = 10, 0
    for x in nums:
        r = digit_range(x)
        if r < best:
            best, total = r, x
        elif r == best:
            total += x
    return total

assert min_digit_range_sum([29, 92, 55, 33]) == 55 + 33 and min_digit_range_sum([12, 34]) == 12 + 34

# Bài 4
def number_with_max_range(nums):
    best, ans = -1, None
    for x in nums:
        r = digit_range(x)
        if r > best:
            best, ans = r, x
    return ans

assert number_with_max_range([15, 29, 92]) == 29 and number_with_max_range([11, 22]) == 11

# Bài 5
def count_max_range(nums):
    best, cnt = -1, 0
    for x in nums:
        r = digit_range(x)
        if r > best:
            best, cnt = r, 1
        elif r == best:
            cnt += 1
    return cnt

assert count_max_range([29, 92, 55]) == 2 and count_max_range([11, 22, 33]) == 3
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def maxDigitRange(self, nums: List[int]) -> int:
        best, total = -1, 0
        for x in nums:
            s = str(x)
            r = int(max(s)) - int(min(s))   # digit range
            if r > best:
                best, total = r, x           # nhóm tốt hơn: ĐẶT LẠI tổng
            elif r == best:
                total += x                   # cùng nhóm tốt nhất: cộng thêm
        return total
```

Khung chung "tổng/đếm các phần tử đạt giá trị lớn nhất của `f`": `best`, `total`, đặt lại khi `f > best`.

### Checklist

- [ ] Đã đọc kỹ constraints (độ dài, miền giá trị) và biết thuật toán chạy được trong giới hạn đó.
- [ ] Đã chạy tay ít nhất **một ví dụ của đề** và một **trường hợp biên** (nhỏ nhất, lớn nhất, giá trị đặc biệt).
- [ ] Đúng **tên class / tên hàm / kiểu trả về** như skeleton LeetCode (đúng `bool`, `int`, `list`...).
- [ ] Không dùng `input()`, không `print` thừa trong code nộp.
- [ ] Không làm hỏng dữ liệu đầu vào nếu đề không cho phép (sao chép trước khi sửa).
- [ ] Đã so sánh với một cách brute force trên input ngẫu nhiên nhỏ.
- [ ] Khi `r > best` đã **đặt lại** `total = x` (không chỉ cập nhật `best`); `best` khởi tạo `-1`.
- [ ] Cộng **số** `x`, không cộng range; đã thử mảng một phần tử và các số cùng range.